## BASELINES

In [88]:
import numpy as np
import pandas as pd

from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, KFold

**Examine training data**

In [89]:
# constants across steps
SEED = 42

In [90]:
de_train_path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/data/de_train.parquet"
de_train = pd.read_parquet(de_train_path)
de_train.head()

,cell_type,sm_name,sm_lincs_id,SMILES,control,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1
0,NK cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.104720,-0.077524,-1.625596,-0.144545,0.143555,...,-0.227781,-0.010752,-0.023881,0.674536,-0.453068,0.005164,-0.094959,0.034127,0.221377,0.368755
1,T cells CD4+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.915953,-0.884380,0.371834,-0.081677,-0.498266,...,-0.494985,-0.303419,0.304955,-0.333905,-0.315516,-0.369626,-0.095079,0.704780,1.096702,-0.869887
2,T cells CD8+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,-0.387721,-0.305378,0.567777,0.303895,-0.022653,...,-0.119422,-0.033608,-0.153123,0.183597,-0.555678,-1.494789,-0.213550,0.415768,0.078439,-0.259365
3,T regulatory cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.232893,0.129029,0.336897,0.486946,0.767661,...,0.451679,0.704643,0.015468,-0.103868,0.865027,0.189114,0.224700,-0.048233,0.216139,-0.085024
4,NK cells,Mometasone Furoate,LSM-3349,C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...,False,4.290652,-0.063864,-0.017443,-0.541154,0.570982,...,0.758474,0.510762,0.607401,-0.123059,0.214366,0.487838,-0.819775,0.112365,-0.122193,0.676629


In [91]:
de_train.shape

(614, 18216)

In [92]:
de_train.cell_type.value_counts()

cell_type
NK cells              146
T cells CD4+          146
T regulatory cells    146
T cells CD8+          142
B cells                17
Myeloid cells          17
Name: count, dtype: int64

In [93]:
de_train.sm_name.value_counts()

sm_name
R428                   6
MLN 2238               6
Dactolisib             6
LDN 193189             6
Porcn Inhibitor III    6
                      ..
Masitinib              4
GSK-1070916            4
Isoniazid              4
Resminostat            3
CGP 60474              3
Name: count, Length: 146, dtype: int64

In [94]:
de_train.control.value_counts()

control
False    602
True      12
Name: count, dtype: int64

**Establish Baselines**

Return the average gene expression across cell type.

In [95]:
def mrrmse(y, y_pred):
    return root_mean_squared_error(y.T, y_pred.T)

In [96]:
# implement MeanBaseline
class MeanBaseline:
    def fit(self, X, y):
        self.means = y.mean()   # average gene expression PER GENE

    def predict(self, X):
        df = pd.DataFrame([self.means] * X.shape[0])
        return df

# implement MedianBaseline
class MedianBaseline:
    def fit(self, X, y):
        self.median = y.median()    # median gene expression PER GENE

    def predict(self, X):
        df = pd.DataFrame([self.median] * X.shape[0])
        return df

# implement ConditionalMeanBaseline
class ConditionalMeanBaseline:
    def __init__(self, condition='sm_name'):
        self.condition = condition

    def fit(self, X, y):
        if self.condition not in X.columns:
            raise ValueError("condition must be in training set </3")
        self.conditional_mean = {} # initialize lookup table category -> mean

        # Get unique categories in `condition`
        u_cat = X[self.condition].unique()

        # For each category, get the mean of that category across samples
        for cat in u_cat:
            is_cat = X[self.condition].values == cat
            cat_mean = y.values[is_cat].mean(axis=0, keepdims=True) #(1, N)
            self.conditional_mean[cat] = cat_mean
        # print(self.conditional_mean.keys())

    def predict(self, X):
        if self.condition not in X.columns:
            raise ValueError("condition must be in test set")

        default_shape = list(self.conditional_mean.values())[0].size
        # print(default_shape)
        y_hat = []

        for cat in X[self.condition].values:
            #  print(cat)
            if cat not in self.conditional_mean.keys():
                # print(f'{cat} not in training set')
                y_hat.append(np.zeros(default_shape))
            else: 
                # print(list(self.conditional_mean[cat]))
                y_hat.append(list(self.conditional_mean[cat]))
        # now u finish :)
        df = pd.DataFrame(np.vstack(y_hat))
        return df


In [97]:
# predict by returning mean gene expression 
kfold = KFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
mrrmse_list = []

for i, (train, test) in enumerate(kfold.split(de_train)):
    model = MeanBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.7814931743754341, MRRMSE = 1.1657943518916691
fold 1, r2 score = -0.5333020984338789, MRRMSE = 1.6101826673064017
fold 2, r2 score = -0.6008189088871045, MRRMSE = 1.1370126589107823
fold 3, r2 score = -0.42680356910025324, MRRMSE = 1.6033063708464486
fold 4, r2 score = -0.42181938114659334, MRRMSE = 1.1636169096047184
average mrrmse: 1.335982591712004


In [98]:
# predict by returning mean gene expression, stratified by drug
kfold = StratifiedKFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
strat_labels = de_train["sm_name"]
mrrmse_list = []

for train, test in kfold.split(de_train, strat_labels):
    model = MeanBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

/Users/jennyng/miniforge3/envs/bioml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


r2 score = -0.5084326345643877, MRRMSE = 1.4034054644658713
r2 score = -0.5177720120361836, MRRMSE = 1.348595597505675
r2 score = -0.4972054770481811, MRRMSE = 1.381639670087728
r2 score = -0.6763956726382737, MRRMSE = 1.2662860216956864
r2 score = -0.5370814292091233, MRRMSE = 1.2732128796313242
average mrrmse: 1.334627926677257


In [99]:
# predict by returning median gene expression 
kfold = KFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
mrrmse_list = []

for i, (train, test) in enumerate(kfold.split(de_train)):
    model = MedianBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.12554377609234196, MRRMSE = 1.0442198760608212
fold 1, r2 score = -0.13599085458795196, MRRMSE = 1.5445427517591779
fold 2, r2 score = -0.11068893020560593, MRRMSE = 1.0490851192527342
fold 3, r2 score = -0.15483080491109577, MRRMSE = 1.575278078493714
fold 4, r2 score = -0.07664850755465825, MRRMSE = 1.1169271635973759
average mrrmse: 1.2660105978327647


In [100]:
# predict by returning median gene expression, stratified by drug
kfold = StratifiedKFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
strat_labels = de_train["sm_name"]
mrrmse_list = []

for train, test in kfold.split(de_train, strat_labels):
    model = MedianBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

/Users/jennyng/miniforge3/envs/bioml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


r2 score = -0.1603304296469136, MRRMSE = 1.3617642037778048
r2 score = -0.12182423264933188, MRRMSE = 1.2884963242776664
r2 score = -0.1155156671394374, MRRMSE = 1.3218213095974074
r2 score = -0.09534035266026565, MRRMSE = 1.151314152944591
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
average mrrmse: 1.265930060224322


In [101]:
# predict by returning median gene expression, LOGO
logo = LeaveOneGroupOut()
features = ["cell_type", "sm_name"]
groups = de_train["cell_type"]
mrrmse_list = []
logo.get_n_splits(groups=groups)    # 6

for i, (train_idx, test_idx) in enumerate(logo.split(de_train, groups=groups)):
    model = MedianBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
r2 score = -0.10651097963195794, MRRMSE = 1.2062543105241406
average mrrmse: 1.2062543105241408


In [102]:
# predict by returning conditional mean gene expression of drug
kfold = KFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
mrrmse_list = []
for i, (train, test) in enumerate(kfold.split(de_train)):
    model = ConditionalMeanBaseline('sm_name')
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.7185879003413586, MRRMSE = 1.136695953022008
fold 1, r2 score = -0.9194269530983071, MRRMSE = 1.6679694917533634
fold 2, r2 score = -0.702369343392255, MRRMSE = 1.2560362614885583
fold 3, r2 score = -0.6898363142477788, MRRMSE = 1.637720774019663
fold 4, r2 score = -0.4249516155958574, MRRMSE = 1.123365507449906
average mrrmse: 1.3643575975466997


In [103]:
# predict by returning conditional mean gene expression of drug, stratified by drug
kfold = StratifiedKFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
strat_labels = de_train["sm_name"]
mrrmse_list = []

for train, test in kfold.split(de_train, strat_labels):
    model = ConditionalMeanBaseline('sm_name')
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

r2 score = -0.6230030969958871, MRRMSE = 1.446607062739374
r2 score = -0.5059690593483391, MRRMSE = 1.2405699303632445


/Users/jennyng/miniforge3/envs/bioml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


r2 score = -0.5795168660490457, MRRMSE = 1.351714515885192
r2 score = -0.5169488742215851, MRRMSE = 1.2500966928268094
r2 score = -0.6431004575625773, MRRMSE = 1.2697978733630997
average mrrmse: 1.3117572150355439


In [104]:
kfold = KFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
mrrmse_list = []

for i, (train, test) in enumerate(kfold.split(de_train)):
    model = ConditionalMeanBaseline('cell_type')
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -1.459227418273887, MRRMSE = 1.2571563254198754
fold 1, r2 score = -0.5354832780048993, MRRMSE = 1.591072139997602
fold 2, r2 score = -1.0244984432065602, MRRMSE = 1.1740315731431572
fold 3, r2 score = -1.155623310339163, MRRMSE = 1.6555857749646332
fold 4, r2 score = -0.4626282962419072, MRRMSE = 1.1481281345417031
average mrrmse: 1.3651947896133942


In [105]:
# predict by returning conditional mean gene expression of cell type, stratified by drug
kfold = StratifiedKFold(shuffle=True, random_state=SEED)
features = ["cell_type", "sm_name"]
strat_labels = de_train["sm_name"]
mrrmse_list = []

for train, test in kfold.split(de_train, strat_labels):
    model = ConditionalMeanBaseline('cell_type')
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

r2 score = -0.5533239773273014, MRRMSE = 1.3777904830078016
r2 score = -0.8816434406547675, MRRMSE = 1.4003183662862575


/Users/jennyng/miniforge3/envs/bioml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


r2 score = -0.7132446796858318, MRRMSE = 1.390054888535577
r2 score = -0.9774310901475904, MRRMSE = 1.2859933300543447
r2 score = -0.9491291246761644, MRRMSE = 1.3033951266099122
average mrrmse: 1.3515104388987786


**Leave One Group Out**

In [106]:
hold_out = ["B cells", "Myeloid cells"]
de_train["cell_type"].isin(hold_out)

0      False
1      False
2      False
3      False
4      False
       ...  
609    False
610    False
611    False
612    False
613    False
Name: cell_type, Length: 614, dtype: bool

In [107]:
subset = de_train[de_train["cell_type"].isin(hold_out)]
subset.head()

,cell_type,sm_name,sm_lincs_id,SMILES,control,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1
8,B cells,Idelalisib,LSM-1205,CC[C@H](Nc1ncnc2[nH]cnc12)c1nc2cccc(F)c2c(=O)n...,False,0.394173,-0.153824,0.178232,0.566241,0.391377,...,-1.052302,-1.176587,-1.220291,-0.278944,-0.095066,1.101790,0.061803,1.406335,-0.264996,-0.119743
9,Myeloid cells,Idelalisib,LSM-1205,CC[C@H](Nc1ncnc2[nH]cnc12)c1nc2cccc(F)c2c(=O)n...,False,0.025146,0.316388,1.366885,1.301593,2.317619,...,-0.902546,-1.445523,0.794772,0.428973,0.605834,0.271988,0.492231,0.354721,1.471559,-0.259483
30,B cells,Crizotinib,LSM-1027,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,False,-0.221234,-0.588647,0.021279,1.104213,1.584397,...,-0.363383,0.215196,1.177265,0.832644,0.296236,0.603842,1.152725,0.599055,0.253165,0.762857
31,Myeloid cells,Crizotinib,LSM-1027,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,False,1.322002,-0.083050,1.630350,0.238302,8.488463,...,-1.610365,-0.624159,-0.977755,-1.992494,1.265720,0.461353,-0.270218,-0.077017,-0.508337,0.454613
72,B cells,Linagliptin,LSM-45916,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nc(C)...,False,-0.096688,0.057827,0.261704,0.054919,1.010873,...,-0.130766,-0.990278,-1.384846,0.742082,-0.349105,-0.091224,-0.721902,-0.098806,-1.570230,-0.013876


In [108]:
subset["sm_name"].value_counts()

sm_name
Idelalisib                  2
Belinostat                  2
Oprozomib (ONX 0912)        2
O-Demethylated Adapalene    2
Dactolisib                  2
Penfluridol                 2
MLN 2238                    2
Foretinib                   2
Porcn Inhibitor III         2
Crizotinib                  2
R428                        2
LDN 193189                  2
Alvocidib                   2
Dabrafenib                  2
Palbociclib                 2
Linagliptin                 2
CHIR-99021                  2
Name: count, dtype: int64

In [109]:
# predict by returning median gene expression, LOGO
logo = LeaveOneGroupOut()
features = ["cell_type", "sm_name"]
groups = de_train["cell_type"]
mrrmse_list = []
logo.get_n_splits(groups=groups)    # 6

for i, (train_idx, test_idx) in enumerate(logo.split(de_train, groups=groups)):
    model = MedianBaseline()
    X_train = de_train.loc[train, features]
    y_train = de_train.iloc[train, 5:]
    X_test = de_train.loc[test, features]
    y_test = de_train.iloc[test, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    held_out_group = np.unique(groups[test_idx])[0]

    print(f"held out group: {held_out_group}, MRRMSE = {mrrmse(y_test, y_pred)}")
    mrrmse_list.append(mrrmse(y_test, y_pred))

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

held out group: B cells, MRRMSE = 1.2062543105241406
held out group: Myeloid cells, MRRMSE = 1.2062543105241406
held out group: NK cells, MRRMSE = 1.2062543105241406
held out group: T cells CD4+, MRRMSE = 1.2062543105241406
held out group: T cells CD8+, MRRMSE = 1.2062543105241406
held out group: T regulatory cells, MRRMSE = 1.2062543105241406
average mrrmse: 1.2062543105241408


**PICKLE SPLITZZ**

Re-assessing baselines using new splits

In [110]:
import pickle

In [111]:
path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/processed_data/train_test_splits.pkl"
with open(path, "rb") as f:
    folds = pickle.load(f)

In [112]:
mrrmse_list = []

for i, (train_idx, test_idx) in enumerate(folds):
    model = MeanBaseline()
    X_train = de_train.iloc[train_idx].iloc[:, :5]
    y_train = de_train.iloc[train_idx].iloc[:, 5:]
    X_test = de_train.iloc[test_idx].iloc[:, :5]
    y_test = de_train.iloc[test_idx].iloc[:, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    err = mrrmse(y_test, y_pred)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {err}")
    mrrmse_list.append(err)

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.6307851229558987, MRRMSE = 1.3209598271842464
fold 1, r2 score = -0.45755446460115856, MRRMSE = 1.1868509100361975
fold 2, r2 score = -0.6011380421970031, MRRMSE = 1.6175502815986589
fold 3, r2 score = -0.5590682829957756, MRRMSE = 1.3510822804390379
fold 4, r2 score = -0.38392341844276673, MRRMSE = 1.7410636875131322
fold 5, r2 score = -0.6028166595437408, MRRMSE = 1.1744338322718781
fold 6, r2 score = -0.5513749173420813, MRRMSE = 1.016755056838596
fold 7, r2 score = -0.5533285948760933, MRRMSE = 1.0726439532047967
fold 8, r2 score = -0.6880966569556765, MRRMSE = 1.229475584504458
fold 9, r2 score = -0.4413890805833207, MRRMSE = 1.6165721045585022
average mrrmse: 1.3327387518149503


In [113]:
mrrmse_list = []

for i, (train_idx, test_idx) in enumerate(folds):
    model = MedianBaseline()
    X_train = de_train.iloc[train_idx].iloc[:, :5]
    y_train = de_train.iloc[train_idx].iloc[:, 5:]
    X_test = de_train.iloc[test_idx].iloc[:, :5]
    y_test = de_train.iloc[test_idx].iloc[:, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    err = mrrmse(y_test, y_pred)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {err}")
    mrrmse_list.append(err)

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.16999280658408839, MRRMSE = 1.2437315041715464
fold 1, r2 score = -0.09347435929503738, MRRMSE = 1.122980624195546
fold 2, r2 score = -0.15517272091540737, MRRMSE = 1.5520098978253174
fold 3, r2 score = -0.07784909387927981, MRRMSE = 1.258750698148323
fold 4, r2 score = -0.11869313047754201, MRRMSE = 1.722393451609928
fold 5, r2 score = -0.13757623981610556, MRRMSE = 1.0980687894065877
fold 6, r2 score = -0.07498551315870071, MRRMSE = 0.9350838624715622
fold 7, r2 score = -0.08988383878842605, MRRMSE = 0.9902826697830833
fold 8, r2 score = -0.13501363033822886, MRRMSE = 1.1337707196710751
fold 9, r2 score = -0.14347790806774086, MRRMSE = 1.5820665101797793
average mrrmse: 1.2639138727462746


In [114]:
mrrmse_list = []

for i, (train_idx, test_idx) in enumerate(folds):
    model = ConditionalMeanBaseline('sm_name')
    X_train = de_train.iloc[train_idx].iloc[:, :5]
    y_train = de_train.iloc[train_idx].iloc[:, 5:]
    X_test = de_train.iloc[test_idx].iloc[:, :5]
    y_test = de_train.iloc[test_idx].iloc[:, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    err = mrrmse(y_test, y_pred)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {err}")
    mrrmse_list.append(err)

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -0.7683171555153676, MRRMSE = 1.2294358790461617
fold 1, r2 score = -0.5357648336751661, MRRMSE = 1.228896287081401
fold 2, r2 score = -0.78655976482241, MRRMSE = 1.6803462229376818
fold 3, r2 score = -0.7689340931738806, MRRMSE = 1.3883612059644383
fold 4, r2 score = -0.5449312035827072, MRRMSE = 1.44081456786079
fold 5, r2 score = -0.5398271903877992, MRRMSE = 1.2195944001661698
fold 6, r2 score = -0.5856682829942625, MRRMSE = 1.0517451599333232
fold 7, r2 score = -0.7634835989827576, MRRMSE = 1.1783288048020233
fold 8, r2 score = -0.7633690456187673, MRRMSE = 1.3214182734920774
fold 9, r2 score = -0.6836251921622764, MRRMSE = 1.619463783709407
average mrrmse: 1.3358404584993473


In [115]:
mrrmse_list = []

for i, (train_idx, test_idx) in enumerate(folds):
    model = ConditionalMeanBaseline('cell_type')
    X_train = de_train.iloc[train_idx].iloc[:, :5]
    y_train = de_train.iloc[train_idx].iloc[:, 5:]
    X_test = de_train.iloc[test_idx].iloc[:, :5]
    y_test = de_train.iloc[test_idx].iloc[:, 5:]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    err = mrrmse(y_test, y_pred)
    print(f"fold {i}, r2 score = {r2_score(y_test.T, y_pred.T)}, MRRMSE = {err}")
    mrrmse_list.append(err)

avg_mrrmse = sum(mrrmse_list)/len(mrrmse_list)
print(f"average mrrmse: {avg_mrrmse}")

fold 0, r2 score = -1.0019202631534367, MRRMSE = 1.3435588269870102
fold 1, r2 score = -0.5863480403445578, MRRMSE = 1.1715147277997335
fold 2, r2 score = -1.077307569774329, MRRMSE = 1.658911794353903
fold 3, r2 score = -0.8164488220685276, MRRMSE = 1.3403040689849415
fold 4, r2 score = -0.5047499480652303, MRRMSE = 1.7265702163933947
fold 5, r2 score = -1.0329821323119173, MRRMSE = 1.229416875648161
fold 6, r2 score = -0.9025065867292141, MRRMSE = 1.067378494156077
fold 7, r2 score = -0.7821717072818744, MRRMSE = 1.1236522212160858
fold 8, r2 score = -1.1160390882965894, MRRMSE = 1.253438705321743
fold 9, r2 score = -0.4339912739435235, MRRMSE = 1.593028887620699
average mrrmse: 1.350777481848175
